In [1]:


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print("All imports successful")

Device: cpu
PyTorch: 2.12.0
All imports successful


In [2]:
import os
os.chdir('/Users/parthshringarpure/Desktop/AI/Projects/luad_survival')

# Load all four streams
expr     = pd.read_csv('data/processed/expression_matrix.csv', index_col=0)
dysreg   = pd.read_csv('data/processed/dysregulation_scores.csv', index_col=0)
immune   = pd.read_csv('data/processed/immune_features_cibersort.csv', index_col=0)
clinical = pd.read_csv('data/processed/clinical_survival.csv', index_col=0)

# Align all patients
common   = expr.index.intersection(dysreg.index).intersection(immune.index).intersection(clinical.index)
expr     = expr.loc[common]
dysreg   = dysreg.loc[common]
immune   = immune.loc[common]
clinical = clinical.loc[common]

# Clinical features
age           = clinical[['age']].copy()
gender        = (clinical['gender'] == 'male').astype(float).to_frame()
stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features = pd.concat([age, gender, stage_dummies], axis=1).astype(float).fillna(0)

# Survival labels
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

print(f"Patients:          {len(common)}")
print(f"Expression:        {expr.shape}")
print(f"Dysregulation:     {dysreg.shape}")
print(f"Immune:            {immune.shape}")
print(f"Clinical features: {clinical_features.shape}")
print(f"Events:            {y['event'].sum()} ({y['event'].mean()*100:.1f}%)")
print(f"Clinical cols:     {list(clinical_features.columns)}")

Patients:          478
Expression:        (478, 1000)
Dysregulation:     (478, 819)
Immune:            (478, 22)
Clinical features: (478, 5)
Events:            121 (25.3%)
Clinical cols:     ['age', 'gender', 'stage_Stage II', 'stage_Stage III', 'stage_Stage IV']


In [4]:
class FusionModelV3(nn.Module):
    def __init__(self, expr_dim=30, dysreg_dim=20, 
                 immune_dim=22, clinical_dim=5, dropout=0.5):
        super().__init__()
        
        self.encoder_expr = nn.Sequential(
            nn.Linear(expr_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )
        self.encoder_dysreg = nn.Sequential(
            nn.Linear(dysreg_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )
        self.encoder_immune = nn.Sequential(
            nn.Linear(immune_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )
        self.encoder_clinical = nn.Sequential(
            nn.Linear(clinical_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32)
        )
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )
        self.output = nn.Linear(32, 1)
    
    def forward(self, x_expr, x_dysreg, x_immune, x_clinical):
        h_expr     = self.encoder_expr(x_expr)
        h_dysreg   = self.encoder_dysreg(x_dysreg)
        h_immune   = self.encoder_immune(x_immune)
        h_clinical = self.encoder_clinical(x_clinical)
        
        streams      = torch.stack([h_expr, h_dysreg, h_immune, h_clinical], dim=1)
        attn_weights = torch.softmax(self.attention(streams), dim=1)
        fused        = (attn_weights * streams).sum(dim=1)
        
        return self.output(fused), attn_weights.squeeze(-1)


def cox_loss(risk_scores, times, events):
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order].squeeze()
    events      = events[order]
    log_cumsum  = torch.logcumsumexp(risk_scores, dim=0)
    return -torch.mean((risk_scores - log_cumsum)[events.bool()])


class SurvivalDataset(Dataset):
    def __init__(self, expr, dysreg, immune, clinical, times, events):
        self.expr     = torch.FloatTensor(expr)
        self.dysreg   = torch.FloatTensor(dysreg)
        self.immune   = torch.FloatTensor(immune)
        self.clinical = torch.FloatTensor(clinical)
        self.times    = torch.FloatTensor(times)
        self.events   = torch.FloatTensor(events)
    
    def __len__(self):
        return len(self.times)
    
    def __getitem__(self, idx):
        return (self.expr[idx], self.dysreg[idx],
                self.immune[idx], self.clinical[idx],
                self.times[idx], self.events[idx])


def train_model(model, train_loader, val_expr, val_dysreg,
                val_immune, val_clinical, val_times, val_events,
                epochs=300, patience=30, lr=0.001, noise=0.05):
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    best_val_cindex  = 0
    best_weights     = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        for x_expr, x_dysreg, x_immune, x_clinical, times, events in train_loader:
            x_expr     = x_expr.to(device)
            x_dysreg   = x_dysreg.to(device)
            x_immune   = x_immune.to(device)
            x_clinical = x_clinical.to(device)
            times      = times.to(device)
            events     = events.to(device)
            
            # Gaussian noise augmentation on molecular streams only
            x_expr   = x_expr   + torch.randn_like(x_expr)   * noise
            x_dysreg = x_dysreg + torch.randn_like(x_dysreg) * noise
            x_immune = x_immune + torch.randn_like(x_immune) * noise
            
            optimizer.zero_grad()
            risk, _ = model(x_expr, x_dysreg, x_immune, x_clinical)
            loss = cox_loss(risk, times, events)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_risk, _ = model(
                val_expr.to(device), val_dysreg.to(device),
                val_immune.to(device), val_clinical.to(device))
            val_risk = val_risk.squeeze().cpu().numpy()
        
        val_ci = concordance_index_censored(
            val_events.astype(bool), val_times, val_risk)[0]
        
        scheduler.step(-val_ci)
        
        if val_ci > best_val_cindex:
            best_val_cindex  = val_ci
            best_weights     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    model.load_state_dict(best_weights)
    return model, best_val_cindex, epoch


# Quick test
m = FusionModelV3().to(device)
r, a = m(torch.randn(4,30).to(device), torch.randn(4,20).to(device),
          torch.randn(4,22).to(device), torch.randn(4,5).to(device))
print(f"Model test passed — risk: {r.shape}, attn: {a.shape}")
print("All components defined ✅")

Model test passed — risk: torch.Size([4, 1]), attn: torch.Size([4, 4])
All components defined ✅


In [8]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex       = []
fold_attn_weights = []

print("Running leakage-free 5-fold CV — FusionModelV3")
print("Cox gene selection + Gaussian augmentation + StratifiedKFold")
print(f"{'Fold':<6} {'Best Epoch':<12} {'Test C-index':<12}")
print("-" * 32)

for fold, (train_idx, test_idx) in enumerate(kf.split(expr, y['event']), 1):

    expr_train,     expr_test     = expr.iloc[train_idx],              expr.iloc[test_idx]
    dysreg_train,   dysreg_test   = dysreg.iloc[train_idx],            dysreg.iloc[test_idx]
    immune_train,   immune_test   = immune.iloc[train_idx],            immune.iloc[test_idx]
    clinical_train, clinical_test = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    y_train,        y_test        = y[train_idx],                      y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # ── Cox-guided gene selection on training patients only ──────────
    cox_pvals_expr = {}
    for gene in expr_train.columns:
        try:
            df_tmp = pd.DataFrame({'T': times_train, 'E': events_train,
                                   'gene': expr_train[gene].values})
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals_expr[gene] = cph.summary['p'].values[0]
        except:
            cox_pvals_expr[gene] = 1.0
    top_expr_genes = pd.Series(cox_pvals_expr).nsmallest(30).index

    cox_pvals_dysreg = {}
    for gene in dysreg_train.columns:
        try:
            df_tmp = pd.DataFrame({'T': times_train, 'E': events_train,
                                   'gene': dysreg_train[gene].values})
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals_dysreg[gene] = cph.summary['p'].values[0]
        except:
            cox_pvals_dysreg[gene] = 1.0
    top_dysreg_genes = pd.Series(cox_pvals_dysreg).nsmallest(20).index
    # ────────────────────────────────────────────────────────────────

    expr_train_sel   = expr_train[top_expr_genes]
    expr_test_sel    = expr_test[top_expr_genes]
    dysreg_train_sel = dysreg_train[top_dysreg_genes]
    dysreg_test_sel  = dysreg_test[top_dysreg_genes]

    scaler_expr     = StandardScaler()
    scaler_dysreg   = StandardScaler()
    scaler_immune   = StandardScaler()
    scaler_clinical = StandardScaler()

    expr_train_s     = scaler_expr.fit_transform(expr_train_sel)
    expr_test_s      = scaler_expr.transform(expr_test_sel)
    dysreg_train_s   = scaler_dysreg.fit_transform(dysreg_train_sel)
    dysreg_test_s    = scaler_dysreg.transform(dysreg_test_sel)
    immune_train_s   = scaler_immune.fit_transform(immune_train)
    immune_test_s    = scaler_immune.transform(immune_test)
    clinical_train_s = scaler_clinical.fit_transform(clinical_train)
    clinical_test_s  = scaler_clinical.transform(clinical_test)

    val_size     = int(0.2 * len(train_idx))
    val_expr     = torch.FloatTensor(expr_train_s[:val_size])
    val_dysreg   = torch.FloatTensor(dysreg_train_s[:val_size])
    val_immune   = torch.FloatTensor(immune_train_s[:val_size])
    val_clinical = torch.FloatTensor(clinical_train_s[:val_size])
    val_times    = times_train[:val_size].copy()
    val_events   = events_train[:val_size].copy()

    train_dataset = SurvivalDataset(
        expr_train_s, dysreg_train_s, immune_train_s, clinical_train_s,
        times_train.copy(), events_train.copy())
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    model = FusionModelV3(
        expr_dim=30, dysreg_dim=20, immune_dim=22, clinical_dim=5
    ).to(device)

    model, best_val_ci, best_epoch = train_model(
        model, train_loader,
        val_expr, val_dysreg, val_immune, val_clinical,
        val_times, val_events,
        epochs=300, patience=30, lr=0.001, noise=0.02
    )

    model.eval()
    with torch.no_grad():
        test_risk, test_attn = model(
            torch.FloatTensor(expr_test_s).to(device),
            torch.FloatTensor(dysreg_test_s).to(device),
            torch.FloatTensor(immune_test_s).to(device),
            torch.FloatTensor(clinical_test_s).to(device)
        )

    test_risk = test_risk.squeeze().cpu().numpy()
    test_attn = test_attn.cpu().numpy()

    ci_test = concordance_index_censored(
        events_test.astype(bool), times_test, test_risk)[0]

    fold_cindex.append(ci_test)
    fold_attn_weights.append(test_attn)

    print(f"{fold:<6} {best_epoch:<12} {ci_test:.4f}")

print("-" * 32)
print(f"\nFusion Model C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nFull comparison:")
print(f"  Cox Clinical:                    0.700")
print(f"  Cox-Lasso Expression:            0.649")
print(f"  Fusion V3 (no augmentation):     0.655")
print(f"  Fusion V3 (with augmentation):   {np.mean(fold_cindex):.3f}")

Running leakage-free 5-fold CV — FusionModelV3
Cox gene selection + Gaussian augmentation + StratifiedKFold
Fold   Best Epoch   Test C-index
--------------------------------
1      88           0.6712
2      145          0.6189
3      91           0.5519
4      118          0.7470
5      168          0.7444
--------------------------------

Fusion Model C-index: 0.667 ± 0.075

Full comparison:
  Cox Clinical:                    0.700
  Cox-Lasso Expression:            0.649
  Fusion V3 (no augmentation):     0.655
  Fusion V3 (with augmentation):   0.667


In [9]:
save_dir = '/Users/parthshringarpure/Desktop/Projects/luad_survival/models/experiments/v3_aug_noise002'
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), f'{save_dir}/fusion_model_v3_aug002.pt')

results = {
    "model": "FusionModelV3",
    "gene_selection": "univariate Cox p-value",
    "cv_strategy": "StratifiedKFold_5fold",
    "augmentation": "Gaussian noise=0.02",
    "expr_genes": 30,
    "dysreg_genes": 20,
    "cv_cindex_mean": round(float(np.mean(fold_cindex)), 3),
    "cv_cindex_std": round(float(np.std(fold_cindex)), 3),
    "fold_cindices": [round(float(c), 4) for c in fold_cindex]
}

with open(f'{save_dir}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved: {save_dir}")
print(f"C-index: {results['cv_cindex_mean']} ± {results['cv_cindex_std']}")

Saved: /Users/parthshringarpure/Desktop/Projects/luad_survival/models/experiments/v3_aug_noise002
C-index: 0.667 ± 0.075
